# Tujuan, Hipotesis, dan Data

1. Tujuan:
* Mencari apakah variabel X mempengaruhi variabel Y
* Mencari arah dan besarnya pengaruh tersebut

2. Hipotesis:
* H0 = Tidak ada pengaruh antara X dengan Y
* H1 = Ada pengaruh antara X dengan Y

3. Data:
* Data dibuat dengan dummy, tapi menggunakan seed. Jadi akan konsisten walaupun di-run beberapa kali. Masing-masing variabel X dan Y memiliki subbagian menjadi X1, X2, X3, Y1, Y2. Dan masing-masing subbagian juga memiliki nilai yang didapat dari beberapa pertanyaan dan simulasi jawabannya dengan skala Likert.

Lebih disarankan untuk mempelajari terlebih dahulu `Simple-Regresi Linear.ipynb` karena ada panduan lengkap.

# Import dan Create Data

In [1]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy.stats import pearsonr, shapiro
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.diagnostic import het_breuschpagan, linear_rainbow

In [2]:
# ==========================================
# FUNGSI BANTUAN: Menghitung Cronbach's Alpha
# ==========================================
def cronbach_alpha(df):
    df_var = df.var(ddof=1)
    df_total_var = df.sum(axis=1).var(ddof=1)
    n_items = df.shape[1]
    return (n_items / (n_items - 1)) * (1 - (df_var.sum() / df_total_var))

In [3]:
# ==========================================
# TAHAP 0: GENERATE DATA DUMMY (LIKERT 1-5)
# ==========================================

# Seed dan n_respondents bisa diganti jika mau mendapatkan data yang berbeda
np.random.seed(10)
n_respondents = 200

# 1. Buat "Skor Dasar" untuk tiap responden (mewakili karakter aslinya)
# Semakin tinggi skor dasarnya, semakin cenderung dia menjawab 4 atau 5
base_x1 = np.random.normal(3.8, 0.8, n_respondents)
base_x2 = np.random.normal(3.5, 0.7, n_respondents)
base_x3 = np.random.normal(3.9, 0.9, n_respondents)

# Y dipengaruhi secara positif oleh X1, X2, dan X3
base_y1 = (0.4 * base_x1) + (0.3 * base_x2) + (0.3 * base_x3) + np.random.normal(0, 0.4, n_respondents)
base_y2 = (0.3 * base_x1) + (0.4 * base_x2) + (0.3 * base_x3) + np.random.normal(0, 0.4, n_respondents)

def generate_items(base_score, num_items):
    items = []
    for _ in range(num_items):
        # Tambahkan sedikit variasi (noise) per soal, lalu bulatkan
        item = np.round(base_score + np.random.normal(0, 0.5, n_respondents))
        # Pastikan tidak ada angka di luar skala Likert 1-5
        item = np.clip(item, 1, 5).astype(int)
        items.append(item)
    return items

# 2. Buat kolom data
data = {}

# Masukkan item X1, X2, X3
for i, item in enumerate(generate_items(base_x1, 3), 1): data[f'X1_{i}'] = item
for i, item in enumerate(generate_items(base_x2, 3), 1): data[f'X2_{i}'] = item
for i, item in enumerate(generate_items(base_x3, 3), 1): data[f'X3_{i}'] = item

# Masukkan item Y1, Y2
for i, item in enumerate(generate_items(base_y1, 3), 1): data[f'Y1_{i}'] = item
for i, item in enumerate(generate_items(base_y2, 3), 1): data[f'Y2_{i}'] = item

# 3. Jadikan DataFrame dan Export ke CSV
df = pd.DataFrame(data)
df.sample(5)

,X1_1,X1_2,X1_3,X2_1,X2_2,X2_3,X3_1,X3_2,X3_3,Y1_1,Y1_2,Y1_3,Y2_1,Y2_2,Y2_3
129,4,4,4,3,3,3,3,1,3,4,3,4,5,5,3
133,4,5,4,4,3,3,3,4,4,4,3,3,3,3,3
72,4,4,5,4,4,3,5,5,5,5,4,5,4,4,3
87,5,4,4,4,4,4,3,4,3,4,4,5,4,4,3
86,4,3,4,4,4,3,5,5,5,4,3,4,5,5,5


In [4]:
# Pengelompokan kolom berdasarkan dimensi
dimensi_cols = {
    'X1': ['X1_1', 'X1_2', 'X1_3'],
    'X2': ['X2_1', 'X2_2', 'X2_3'],
    'X3': ['X3_1', 'X3_2', 'X3_3'],
    'Y_ALL': ['Y1_1', 'Y1_2', 'Y1_3', 'Y2_1', 'Y2_2', 'Y2_3'] # Y digabung jadi satu kesatuan
}

# Uji Validitas dan Reliabilitas (Survei)

In [5]:
print("=== TAHAP 1: UJI VALIDITAS & RELIABILITAS ===")
# Catatan: Karena ini data random, wajar jika hasilnya "Tidak Valid" atau "Tidak Reliabel".
# Fokus pada alur kerjanya.
for nama_dimensi, kolom in dimensi_cols.items():
    print(f"\n--- Menguji Dimensi {nama_dimensi} ---")

    # Hitung Skor Total Dimensi
    skor_total = df[kolom].sum(axis=1)

    # 1. Uji Validitas (Pearson)
    for item in kolom:
        r_stat, p_val = pearsonr(df[item], skor_total)
        status = "Valid" if p_val < 0.05 and r_stat > 0.3 else "Tidak Valid"
        print(f"Validitas {item}: r={r_stat:.3f}, p={p_val:.3f} -> {status}")

    # 2. Uji Reliabilitas (Cronbach's Alpha)
    alpha = cronbach_alpha(df[kolom])
    status_alpha = "Reliabel" if alpha > 0.6 else "Tidak Reliabel"
    print(f"Reliabilitas {nama_dimensi}: Alpha = {alpha:.3f} -> {status_alpha}")

=== TAHAP 1: UJI VALIDITAS & RELIABILITAS ===

--- Menguji Dimensi X1 ---
Validitas X1_1: r=0.847, p=0.000 -> Valid
Validitas X1_2: r=0.879, p=0.000 -> Valid
Validitas X1_3: r=0.872, p=0.000 -> Valid
Reliabilitas X1: Alpha = 0.833 -> Reliabel

--- Menguji Dimensi X2 ---
Validitas X2_1: r=0.852, p=0.000 -> Valid
Validitas X2_2: r=0.836, p=0.000 -> Valid
Validitas X2_3: r=0.847, p=0.000 -> Valid
Reliabilitas X2: Alpha = 0.799 -> Reliabel

--- Menguji Dimensi X3 ---
Validitas X3_1: r=0.859, p=0.000 -> Valid
Validitas X3_2: r=0.875, p=0.000 -> Valid
Validitas X3_3: r=0.882, p=0.000 -> Valid
Reliabilitas X3: Alpha = 0.842 -> Reliabel

--- Menguji Dimensi Y_ALL ---
Validitas Y1_1: r=0.668, p=0.000 -> Valid
Validitas Y1_2: r=0.735, p=0.000 -> Valid
Validitas Y1_3: r=0.677, p=0.000 -> Valid
Validitas Y2_1: r=0.639, p=0.000 -> Valid
Validitas Y2_2: r=0.682, p=0.000 -> Valid
Validitas Y2_3: r=0.614, p=0.000 -> Valid
Reliabilitas Y_ALL: Alpha = 0.754 -> Reliabel


(*) Jika Uji Validitas atau Reliabilitas ada yang tidak lolos, maka yang bisa dilakukan adalah:

1. Drop. Jika ada pertanyaan tidak valid, maka bisa langsung di-drop saja.
2. Parafrase kalimat pertanyaan. Responden mungkin kebingungan dengan maksud pertanyaan tersebut
3. Cek Data. Mungkin saja responden mengisi asal-asalan. Bisa dengan cara diminta isi ulang atau drop saja data responden tersebut.
4. Menambah jumlah responden.

# Agregasi Data

Menggabungkan (agregasi) data dari 1 subbagian (pembagian dari variabel) yang terdiri dari 3 pertanyaan masing-masing menjadi 1 nilai menggunakan rata-rata.

In [6]:
print("\n=== TAHAP 2: AGREGASI DATA (MENGHITUNG RATA-RATA) ===")
# Mengubah data per-item menjadi data per-dimensi (X1, X2, X3, Y)
df_agg = pd.DataFrame()
df_agg['X1'] = df[dimensi_cols['X1']].mean(axis=1)
df_agg['X2'] = df[dimensi_cols['X2']].mean(axis=1)
df_agg['X3'] = df[dimensi_cols['X3']].mean(axis=1)
df_agg['Y']  = df[dimensi_cols['Y_ALL']].mean(axis=1)

print("Data berhasil diagregasi. Contoh 5 data pertama:")
df_agg.head()


=== TAHAP 2: AGREGASI DATA (MENGHITUNG RATA-RATA) ===
Data berhasil diagregasi. Contoh 5 data pertama:


,X1,X2,X3,Y
0,5.000000,3.333333,4.000000,4.666667
1,4.666667,4.666667,2.666667,4.000000
2,2.666667,3.000000,4.000000,3.333333
3,4.000000,3.666667,4.666667,3.833333
4,4.666667,2.333333,3.000000,3.333333


# Uji Asumsi Klasik

Uji Normalitas

In [7]:
print("\n=== TAHAP 3 & 4: PERSIAPAN REGRESI & UJI ASUMSI KLASIK ===")
# Definisikan X dan Y
X = df_agg[['X1', 'X2', 'X3']]
Y = df_agg['Y']
X_const = sm.add_constant(X) # Wajib ditambah konstanta untuk model OLS

# Jalankan Model Regresi untuk mendapatkan Residual
model = sm.OLS(Y, X_const).fit()
residual = model.resid

# A. Uji Normalitas (Shapiro-Wilk)
stat, p_norm = shapiro(residual)
print(f"1. Normalitas (Shapiro-Wilk) p-value: {p_norm:.4f} -> {'Aman (Normal)' if p_norm > 0.05 else 'Tidak Normal'}")


=== TAHAP 3 & 4: PERSIAPAN REGRESI & UJI ASUMSI KLASIK ===
1. Normalitas (Shapiro-Wilk) p-value: 0.4741 -> Aman (Normal)


Uji Multikolinearitas

In [8]:
# B. Uji Multikolinearitas (VIF)
print("2. Multikolinearitas (VIF):")
for i, col in enumerate(X_const.columns):
    if col != 'const':
        vif = variance_inflation_factor(X_const.values, i)
        print(f"   - {col}: {vif:.2f} -> {'Aman' if vif < 10 else 'Terjadi Multikolinearitas'}")

2. Multikolinearitas (VIF):
   - X1: 1.00 -> Aman
   - X2: 1.02 -> Aman
   - X3: 1.02 -> Aman


(*) Jika Uji Multikolinearitas gagal, maka bisa melakukan:
1. Drop Variabel yang gagal
2. Menyatukan variabel tersebut ke variabel lain, bisa menggunakan jumlah atau rata-rata
3. Principal Component Analysis (PCA)
4. Menggunakan Ridge atau Lasso Regression

Uji Heteroskedastisitas

In [9]:
# C. Uji Heteroskedastisitas (Breusch-Pagan)
bp_test = het_breuschpagan(residual, X_const)
print(f"3. Heteroskedastisitas (Breusch-Pagan) p-value: {bp_test[1]:.4f} -> {'Aman' if bp_test[1] > 0.05 else 'Terjadi Heteroskedastisitas'}")

3. Heteroskedastisitas (Breusch-Pagan) p-value: 0.4179 -> Aman


Uji Linearitas

In [10]:
# D. Uji Linearitas (Rainbow Test)
rainbow_stat, rainbow_p = linear_rainbow(model)
print(f"4. Linearitas (Rainbow Test) p-value: {rainbow_p:.4f} -> {'Aman (Linear)' if rainbow_p > 0.05 else 'Tidak Linear'}")

4. Linearitas (Rainbow Test) p-value: 0.5872 -> Aman (Linear)


# Uji Hipotesis

In [11]:
print("\n=== TAHAP 5: HASIL UJI HIPOTESIS (REGRESI BERGANDA) ===")
# Lihat bagian P>|t| untuk parsial, dan Prob (F-statistic) untuk simultan
print(model.summary())


=== TAHAP 5: HASIL UJI HIPOTESIS (REGRESI BERGANDA) ===
                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.443
Model:                            OLS   Adj. R-squared:                  0.435
Method:                 Least Squares   F-statistic:                     51.99
Date:                Tue, 07 Apr 2026   Prob (F-statistic):           9.10e-25
Time:                        06:05:19   Log-Likelihood:                -99.135
No. Observations:                 200   AIC:                             206.3
Df Residuals:                     196   BIC:                             219.5
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------

# Jawaban

* Variabel X mempengaruhi Y secara keseluruhan, dibuktikan dengan nilai Uji F (Prob F-statistic) berada di bawah 0.05, yang berarti Signifikan. Ini berarti kita menolak H0 (X tidak mempengaruhi Y) menjadi H1 (X mempengaruhi Y)
* Jika kita melihat kolom P>|t| atau Uji T masing-masing, seluruh X1, X2, dan X3 memiliki nilai 0.000 atau di bawah 0.05. Hal ini berarti seluruh subbagian atau dimensi X signifikan terhadap Y.
* Subbagian yang paling berpengaruh bisa dilihat dari nilai Koefisiennya (Coef) dan diurutkan dari paling berpengaruh ke tidak yaitu: X1, X2, X3. Dan hubungan masing-masing ketiganya ke Y adalah positif. Jadi jika X1 atau X2 atau X3 tinggi, maka Y juga akan tinggi.
* Nilai Adjusted R-Squared adalah 0.435 yang berarti variabel X (yang terdiri dari X1, X2, X3) dapat menjelaskan 43.5% variasi yang terjadi pada Y. Sisanya 56.5% dijelaskan oleh faktor lain di luar model ini.